# Judge the pipeline's feedback with Qwen3.5-27B

Scores the feedback from `cmp_feedback_base` and `cmp_feedback_adapter` with the feedback repo's judge: `Qwen/Qwen3.5-27B-FP8` on vLLM, thinking on, with the four checks of `feedback/feedback_teacher_qwen3.5-27B.ipynb` (covers missing points, respects met points, faithful to the student, factually correct) and the format checks of `feedback/qlora_feedback_qwen3.5-9B.ipynb`. The prompts and checks are in `src/gfv/judge.py`, copied from those notebooks.

It judges the feedback before rewrites (`initial`) and after the 9B verifier's rewrites (`final`), each against two sets of marks:

- **given**: the marks the feedback was written for (the QLoRA grader's). This measures the feedback itself.
- **gold**: the dataset's marks, as the feedback repo's reference (base 79.7%, adapter 89.7%) used. Feedback that follows a wrong grade fails, so this is the end-to-end result.

Where the two sets of marks agree, and where a rewrite left the feedback unchanged, the prompts are identical and are judged once.

**Before you run it**

- **A 48 GB GPU** (an L40S). The FP8 weights are about 29 GB. Shut down other notebooks first (`run_comparison.ipynb` holds the GPU while its kernel runs).
- **A separate environment with vLLM**, because vLLM installs its own PyTorch. Once, in a terminal on the cluster:

  ```bash
  python3 -m venv /common/scratch/projectgrps/CS480/CS480G3/venv-vllm
  /common/scratch/projectgrps/CS480/CS480G3/venv-vllm/bin/pip install "vllm==0.31.0" pandas scikit-learn scipy ipykernel
  /common/scratch/projectgrps/CS480/CS480G3/venv-vllm/bin/python -m ipykernel install --user --name vllm-judge --display-name "vLLM judge"
  ```

  Then open this notebook with the **vLLM judge** kernel.
- **The finished feedback runs** in `outputs/comparison/cmp_feedback_base/` and `outputs/comparison/cmp_feedback_adapter/`.

**Run All.** Judgements are saved after every chunk to `outputs/comparison/judge_27b/judgements.jsonl`; if the session ends, Run All again and it continues. With nothing left to judge, the model is not loaded, so the results section also runs on a laptop after downloading `judge_27b/` (with the base `requirements.txt`, no vLLM).

## 1. Setup

In [ ]:
import json
import os
import sys

# The experiments repo: the scratch copy on the cluster, else the folder this notebook is in.
SCRATCH_REPO = "/common/scratch/projectgrps/CS480/CS480G3/experiments"
REPO = SCRATCH_REPO if os.path.isdir(SCRATCH_REPO) else os.path.abspath(".")
os.environ.setdefault("HF_HOME", "/common/scratch/projectgrps/CS480/CS480G3/huggingface_cache/")
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import pandas as pd

from src.gfv import judge
from src.gfv.data import load_jsonl
from src.gfv.metrics import load_records

# The feedback runs: outputs/comparison/<config>/ (or outputs/<config>/ in a local download).
RUNS = {"base": "cmp_feedback_base", "adapter": "cmp_feedback_adapter"}
RUN_ROOT = next(root for root in ("outputs/comparison", "outputs")
                if all(os.path.exists(os.path.join(root, name, "records.jsonl")) for name in RUNS.values()))
OUTPUT_DIR = os.path.join(RUN_ROOT, "judge_27b")
JUDGEMENTS_PATH = os.path.join(OUTPUT_DIR, "judgements.jsonl")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SPLIT_SEED = 42           # the feedback runs' test split
CHUNK_SIZE = 512          # judge calls per vLLM batch; saved after each
GPU_MEMORY_UTILIZATION = 0.90
MAX_NUM_SEQS = 128        # Qwen3.5 needs one Mamba cache block per sequence; lower it on a "Mamba cache blocks" error
SEED = 42
N_BOOT = 2000

print("Working in", REPO)
print("Runs from", RUN_ROOT, "->", OUTPUT_DIR)

In [ ]:
df = judge.load_dataset("data/training_data_moh_eng_rubrics.csv")
samples = load_jsonl("data/processed/data.jsonl")
# dataset_index in the records is the row position in the CSV; check they line up.
assert all(samples[i].question.strip() == df.at[i, "question"].strip() for i in range(len(df)))

runs = {short: load_records(os.path.join(RUN_ROOT, name, "records.jsonl")) for short, name in RUNS.items()}
for short, records in runs.items():
    print(f"{short}: {len(records)} rows, {sum(bool(r.get('feedback')) for r in records.values())} with feedback")

# Feedback that used every token the feedback stage allowed counts as truncated.
with open(os.path.join(RUN_ROOT, RUNS["base"], "config_resolved.yaml"), encoding="utf-8") as handle:
    import yaml
    MAX_FEEDBACK_TOKENS = int(yaml.safe_load(handle)["stages"]["feedback"]["decoding"]["max_new_tokens"])
from transformers import AutoTokenizer
feedback_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-9B")

items = judge.build_items(runs, df, lambda text: len(feedback_tokenizer.encode(text, add_special_tokens=False)),
                          MAX_FEEDBACK_TOKENS)
judgements = judge.load_judgements(JUDGEMENTS_PATH)
unique = {item["key"]: item for item in items}
pending = [item for key, item in unique.items() if key not in judgements]
print(f"{len(items)} judgements needed, {len(unique)} distinct prompts, {len(pending)} still to judge")

## 2. Judge

Loads the 27B judge only when something is left to judge. The settings are the teacher notebook's: FP8 weights, thinking on, temperature 0.6, up to 8,192 tokens per judgement. Each chunk of 512 takes several minutes.

In [ ]:
import time

if pending:
    # FlashInfer's sampler needs CUDA >= 12.8; the teacher notebook turns it off for the cluster's CUDA 12.4.
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    try:
        import torch
        from vllm import LLM, SamplingParams
    except ImportError as error:
        raise RuntimeError("vLLM is not installed in this kernel. Pick the 'vLLM judge' kernel "
                           "(see the first cell for how to create it).") from error

    if not torch.cuda.is_available():
        raise RuntimeError("No GPU. Start mynotebook with 'GPU enabled notebook'.")
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name}, {properties.total_memory / 1e9:.0f} GB")
    if properties.total_memory / 1e9 < 44:
        print("Warning: the 27B FP8 judge was set up for a 48 GB L40S and may not fit on this GPU.")

    llm = LLM(
        model=judge.JUDGE_MODEL,
        max_model_len=judge.MAX_MODEL_LEN,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        max_num_seqs=MAX_NUM_SEQS,
        limit_mm_per_prompt={"image": 0, "video": 0},  # text-only use of a vision-language checkpoint
        seed=SEED,
    )
    judge_params = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, max_tokens=judge.MAX_JUDGE_TOKENS, seed=SEED)

    for start in range(0, len(pending), CHUNK_SIZE):
        chunk = pending[start:start + CHUNK_SIZE]
        started = time.perf_counter()
        outputs = llm.chat([item["messages"] for item in chunk], judge_params,
                           chat_template_kwargs={"enable_thinking": judge.JUDGE_THINKING}, use_tqdm=True)
        with open(JUDGEMENTS_PATH, "a", encoding="utf-8") as handle:
            for item, output in zip(chunk, outputs):
                completion = output.outputs[0]
                text = completion.text.split("</think>")[-1].strip()
                record = judge.judgement_record(item["key"], text, completion.finish_reason)
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"Judged {start + len(chunk)}/{len(pending)} in {time.perf_counter() - started:.0f}s", flush=True)
    judgements = judge.load_judgements(JUDGEMENTS_PATH)
else:
    print("Everything is judged; the model is not loaded.")

## 3. Results

Pass rates per system and marks, pass rates by outcome, paired differences with the comparison report's bootstrap by question and McNemar test, and how often the 27B judge agrees with the pipeline's 9B verifier. Written to `judge_27b/report.md` with a CSV for each table.

In [ ]:
from IPython.display import Markdown, display

results = judge.results_frame(items, judgements)
missing = len(items) - len(results)
if missing:
    print(f"Warning: {missing} judgements are missing; run section 2 again.")
tables = judge.summary_tables(results)
paired = judge.paired_tables(results, samples, SPLIT_SEED, N_BOOT)
report_path = judge.write_report(OUTPUT_DIR, results, tables, paired, N_BOOT)
print("Unreadable judgements:", int((~results.parsed).sum()), "of", len(results))
with open(report_path, encoding="utf-8") as handle:
    display(Markdown(handle.read()))